# 🔬 Colab A — NumPy From-Scratch 3-Layer Deep Neural Network
## Nonlinear Regression with Manual Backpropagation & `tf.einsum`

### Key Design Decisions
| Item | Choice |
|---|---|
| Framework | **NumPy** (weights/math) + **tf.einsum** (matrix multiply) |
| Architecture | Input(3) → Hidden1(16) → Hidden2(8) → Output(1) |
| Activation | **Swish** (hidden), Linear (output) |
| Gradient | Full **manual backprop** + chain rule |
| Data | 3-variable synthetic nonlinear: `y = sin(x1) * cos(x2) + x3²` |
| Plot | 4D visualization via PCA (scikit-learn) |


In [ ]:
# ── SECTION 1: Imports ──────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow version:', tf.__version__)

In [ ]:
# ── SECTION 2: Synthetic Data Generation ────────────────────────────────────
# 3-variable nonlinear equation:
#   y = sin(x1) * cos(x2) + x3^2 + 0.1 * noise

N = 1000  # samples
X1 = np.random.uniform(-np.pi, np.pi, N)
X2 = np.random.uniform(-np.pi, np.pi, N)
X3 = np.random.uniform(-2, 2, N)
noise = np.random.randn(N) * 0.1

Y = np.sin(X1) * np.cos(X2) + X3**2 + noise

# Stack into matrix [N, 3]
X = np.column_stack([X1, X2, X3])

# Normalize inputs
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X_norm = scaler_X.fit_transform(X)
Y_norm = scaler_Y.fit_transform(Y.reshape(-1, 1))

print(f'X shape: {X_norm.shape}, Y shape: {Y_norm.shape}')
print(f'Y range: [{Y.min():.3f}, {Y.max():.3f}]')

In [ ]:
# ── SECTION 3: 4D Plot (PCA to reduce 3 inputs → 2D, color = y) ─────────────
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_norm)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], Y_norm.ravel(),
                c=Y_norm.ravel(), cmap='viridis', alpha=0.6, s=8)
plt.colorbar(sc, ax=ax, label='Normalized y')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_zlabel('Target y')
ax.set_title('4D Data Visualization\n(PCA for 3 inputs + color=target)')
plt.tight_layout()
plt.show()
print(f'Explained variance by PCA: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# ── SECTION 4: Weight Initialization ────────────────────────────────────────
# Architecture: 3 → 16 → 8 → 1
def initialize_weights(layer_dims):
    """He initialization for all layers."""
    params = {}
    L = len(layer_dims)
    for l in range(1, L):
        params[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * np.sqrt(2.0 / layer_dims[l-1])
        params[f'b{l}'] = np.zeros((1, layer_dims[l]))
    return params

layer_dims = [3, 16, 8, 1]
params = initialize_weights(layer_dims)
for k, v in params.items():
    print(f'{k}: {v.shape}')

In [ ]:
# ── SECTION 5: Activation Functions ─────────────────────────────────────────
def swish(z):
    """Swish: x * sigmoid(x) — smooth nonlinear activation."""
    return z / (1 + np.exp(-z))

def swish_derivative(z):
    """Derivative of Swish for backprop."""
    sig = 1 / (1 + np.exp(-z))
    return sig + z * sig * (1 - sig)

# Quick sanity check
z_test = np.linspace(-5, 5, 100)
plt.figure(figsize=(8, 3))
plt.plot(z_test, swish(z_test), label='Swish')
plt.plot(z_test, swish_derivative(z_test), label="Swish'")
plt.legend(); plt.title('Swish Activation & Derivative'); plt.grid(True)
plt.show()

In [ ]:
# ── SECTION 6: Forward Pass using tf.einsum ──────────────────────────────────
# tf.einsum replaces np.dot / @ for matrix multiplication
# 'ij,jk->ik' means: for each sample i, dot each input j with weight jk

def forward_pass(X, params):
    """3-layer forward pass. Returns activations cache for backprop."""
    cache = {}

    # Convert to tensors for einsum
    A0 = tf.constant(X, dtype=tf.float32)
    W1 = tf.constant(params['W1'], dtype=tf.float32)
    W2 = tf.constant(params['W2'], dtype=tf.float32)
    W3 = tf.constant(params['W3'], dtype=tf.float32)

    # Layer 1: Input → Hidden1
    Z1_tf = tf.einsum('ij,jk->ik', A0, W1) + params['b1']   # [N,16]
    Z1 = Z1_tf.numpy()
    A1 = swish(Z1)

    # Layer 2: Hidden1 → Hidden2
    A1_tf = tf.constant(A1, dtype=tf.float32)
    Z2_tf = tf.einsum('ij,jk->ik', A1_tf, W2) + params['b2']  # [N,8]
    Z2 = Z2_tf.numpy()
    A2 = swish(Z2)

    # Layer 3: Hidden2 → Output (Linear)
    A2_tf = tf.constant(A2, dtype=tf.float32)
    Z3_tf = tf.einsum('ij,jk->ik', A2_tf, W3) + params['b3']  # [N,1]
    Z3 = Z3_tf.numpy()
    A3 = Z3  # Linear output

    cache = {'A0': X, 'Z1': Z1, 'A1': A1,
             'Z2': Z2, 'A2': A2,
             'Z3': Z3, 'A3': A3}
    return A3, cache

# Test forward pass
y_pred_test, cache_test = forward_pass(X_norm[:5], params)
print('Forward pass output (first 5):', y_pred_test.ravel())

In [ ]:
# ── SECTION 7: Loss Function ─────────────────────────────────────────────────
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

print('Initial MSE:', mse_loss(Y_norm, forward_pass(X_norm, params)[0]))

In [ ]:
# ── SECTION 8: Manual Backpropagation (Chain Rule) ───────────────────────────
#
#  CHAIN RULE FLOW (right to left):
#  dL/dW3 = A2.T @ dZ3
#  dZ2    = (dA2) * swish'(Z2)     where dA2 = dZ3 @ W3.T
#  dZ1    = (dA1) * swish'(Z1)     where dA1 = dZ2 @ W2.T
#
def backward_pass(Y_true, cache, params, N):
    grads = {}
    A0, Z1, A1, Z2, A2, Z3, A3 = (cache['A0'], cache['Z1'],
        cache['A1'], cache['Z2'], cache['A2'], cache['Z3'], cache['A3'])

    # Output layer gradient (MSE loss → linear activation)
    dZ3 = (A3 - Y_true) * (2 / N)       # [N, 1]

    # Layer 3 weights
    grads['dW3'] = A2.T @ dZ3            # [8, 1]
    grads['db3'] = np.sum(dZ3, axis=0, keepdims=True)

    # Backprop into Layer 2
    dA2 = dZ3 @ params['W3'].T           # [N, 8]
    dZ2 = dA2 * swish_derivative(Z2)    # element-wise
    grads['dW2'] = A1.T @ dZ2            # [16, 8]
    grads['db2'] = np.sum(dZ2, axis=0, keepdims=True)

    # Backprop into Layer 1
    dA1 = dZ2 @ params['W2'].T           # [N, 16]
    dZ1 = dA1 * swish_derivative(Z1)
    grads['dW1'] = A0.T @ dZ1            # [3, 16]
    grads['db1'] = np.sum(dZ1, axis=0, keepdims=True)

    return grads

print('Backward pass test OK')

In [ ]:
# ── SECTION 9: Training Loop ─────────────────────────────────────────────────
EPOCHS    = 2000
LR        = 0.01
loss_history = []
N = X_norm.shape[0]

for epoch in range(EPOCHS):
    # Forward
    y_pred, cache = forward_pass(X_norm, params)

    # Loss
    loss = mse_loss(Y_norm, y_pred)
    loss_history.append(loss)

    # Backward
    grads = backward_pass(Y_norm, cache, params, N)

    # Gradient Descent update
    for l in range(1, 4):
        params[f'W{l}'] -= LR * grads[f'dW{l}']
        params[f'b{l}'] -= LR * grads[f'db{l}']

    if epoch % 200 == 0:
        print(f'Epoch {epoch:4d} | MSE Loss: {loss:.6f}')

print(f'\nFinal MSE: {loss_history[-1]:.6f}')

In [ ]:
# ── SECTION 10: Loss Curve & Final Output ────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='royalblue')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training Loss — NumPy 3-Layer Neural Net (Manual Backprop)')
plt.yscale('log'); plt.grid(True)
plt.tight_layout(); plt.show()

# Predictions vs Actual
y_pred_final, _ = forward_pass(X_norm, params)
y_pred_inv = scaler_Y.inverse_transform(y_pred_final)
y_true_inv = scaler_Y.inverse_transform(Y_norm)

plt.figure(figsize=(8, 5))
plt.scatter(y_true_inv[:100], y_pred_inv[:100], alpha=0.6, s=20)
plt.plot([y_true_inv.min(), y_true_inv.max()],
         [y_true_inv.min(), y_true_inv.max()], 'r--', label='Perfect')
plt.xlabel('True y'); plt.ylabel('Predicted y')
plt.title('True vs Predicted — 3-Layer NumPy Net')
plt.legend(); plt.grid(True); plt.show()

final_mse = np.mean((y_true_inv - y_pred_inv)**2)
print(f'Final MSE (original scale): {final_mse:.4f}')